In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('ed38976-edd.csv')

print(" Missing Values Before Cleaning ")
print(df.isnull().sum())

# Handling missing values
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')

print("\nTotal missing values after cleaning:", df.isnull().sum().sum())
le = LabelEncoder()
encoded_cats = ['experience_level', 'learning_style', 'difficulty_level', 'category']
for col in encoded_cats:
    df[col + '_encoded'] = le.fit_transform(df[col])

df.to_csv('ed38976-edd_cleaned.csv', index=False)
print(" Cleaned dataset saved as 'ed38976-edd_cleaned.csv' with shape:", df.shape)

 Missing Values Before Cleaning 
student_id              226
name                    196
age                     202
experience_level        186
learning_style          194
interests               186
course_id               177
course_name             200
category                203
difficulty_level        213
ratings                 217
num_reviews             197
time_spent_on_course    193
completion_status       210
video_id                198
video_topic             193
video_duration          193
time_watched            172
skip_count              217
pause_count             217
disengagement_score     209
dtype: int64

Total missing values after cleaning: 0


/tmp/ipykernel_597/1155185422.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')


 Cleaned dataset saved as 'ed38976-edd_cleaned.csv' with shape: (14101, 25)


Drop out checking

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

df = pd.read_csv('ed38976-edd_cleaned.csv')

# Define Features (X) and Target Variable (y)
# For Dropout Early Warning, predict 'completion_status' (True/False converted to 1/0)
df['dropout_target'] = df['completion_status'].apply(lambda x: 0 if str(x).lower() == 'true' else 1)

# relevant behavioral and demographic features
features = [
    'age', 'time_spent_on_course', 'time_watched',
    'skip_count', 'pause_count', 'disengagement_score',
    'experience_level_encoded', 'learning_style_encoded', 'difficulty_level_encoded'
]

X = df[features]
y = df['dropout_target']

#  Spliting  data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

#Evaluate the model
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

#Save the trained model to disk so it can be used later in FastAPI
joblib.dump(model, 'dropout_warning_model.pkl')
print(" Trained model saved as 'dropout_warning_model.pkl'")

Model Accuracy: 94.33%

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.90      0.93      1111
           1       0.94      0.97      0.95      1710

    accuracy                           0.94      2821
   macro avg       0.94      0.94      0.94      2821
weighted avg       0.94      0.94      0.94      2821

 Trained model saved as 'dropout_warning_model.pkl'


Lead scoring part

was not getting intended results, so currently using rule based

In [12]:
import joblib
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix

# Load your cleaned dataset
df = pd.read_csv('ed38976-edd_cleaned.csv')

#  Define the target
engagement_score = (
    (df['time_watched'] / df['time_watched'].max()) * 0.4
    + (df['ratings'] / 5.0) * 0.4
    - (df['skip_count'] / df['skip_count'].max()) * 0.2
)

threshold = engagement_score.quantile(0.60)
df['lead_converted'] = (engagement_score >= threshold).astype(int)


le_exp = LabelEncoder()
le_diff = LabelEncoder()
le_style = LabelEncoder()
le_cat = LabelEncoder()
df['experience_encoded'] = le_exp.fit_transform(df['experience_level'])
df['difficulty_encoded'] = le_diff.fit_transform(df['difficulty_level'])
df['learning_style_encoded'] = le_style.fit_transform(df['learning_style'])
df['category_encoded'] = le_cat.fit_transform(df['category'])

numeric_features = [
    'age',
    'time_spent_on_course',
    "time_watched",
    "skip_count",
    'pause_count',
    "ratings",
    'num_reviews',
    'video_duration',
    'experience_encoded',
    'difficulty_encoded',
    'learning_style_encoded',
    'category_encoded',
]

X_numeric = df[numeric_features].fillna(0)
y = df['lead_converted']

# Add a small TF-IDF block on 'interests' text for extra signal
tfidf = TfidfVectorizer(stop_words='english', max_features=30)
X_text = tfidf.fit_transform(df['interests'].fillna(''))

X = hstack([csr_matrix(X_numeric.values), X_text])

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tune a RandomForest with GridSearchCV
rf_grid = {
    'n_estimators': [200, 400],
    'max_depth': [8, 12, None],
    'min_samples_leaf': [1, 3, 5],
}
rf_search = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    rf_grid, cv=3, scoring='accuracy', n_jobs=-1
)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

# Trying Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

rf_acc = accuracy_score(y_test, best_rf.predict(X_test))
gb_acc = accuracy_score(y_test, gb.predict(X_test))
print(f'Tuned RandomForest Accuracy: {rf_acc * 100:.2f}%  (best params: {rf_search.best_params_})')
print(f'GradientBoosting Accuracy:  {gb_acc * 100:.2f}%')

best_base_model = best_rf if rf_acc >= gb_acc else gb
calibrated_model = CalibratedClassifierCV(best_base_model, method='sigmoid', cv=3)
calibrated_model.fit(X_train, y_train)

y_pred = calibrated_model.predict(X_test)
print(
    f'\nFinal Calibrated Model Accuracy:'
    f' {accuracy_score(y_test, y_pred) * 100:.2f}%'
)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

joblib.dump(
    {
        'model': calibrated_model,
        'tfidf': tfidf,
        'numeric_features': numeric_features,
        'label_encoders': {
            'experience_level': le_exp,
            'difficulty_level': le_diff,
            'learning_style': le_style,
            'category': le_cat,
        },
    },
    'lead_scoring_model.pkl',
)
print(
    '\nLead Scoring Model (with feature pipeline) saved successfully as'
    " 'lead_scoring_model.pkl'"
)

Tuned RandomForest Accuracy: 96.67%  (best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 400})
GradientBoosting Accuracy:  98.23%

Final Calibrated Model Accuracy: 98.16%

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      1692
           1       0.97      0.98      0.98      1129

    accuracy                           0.98      2821
   macro avg       0.98      0.98      0.98      2821
weighted avg       0.98      0.98      0.98      2821


Lead Scoring Model (with feature pipeline) saved successfully as 'lead_scoring_model.pkl'


Course Recommendation

In [13]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

df = pd.read_csv('ed38976-edd_cleaned.csv')

# Extracting  unique courses and their associated interests/categories
courses_df = df[['course_id', 'course_name', 'category', 'interests']].drop_duplicates().reset_index(drop=True)


courses_df['combined_features'] = courses_df['category'].fillna('') + ' ' + courses_df['interests'].fillna('')

#  Using TF-IDF Vectorizer to convert course descriptions/interests into vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(courses_df['combined_features'])

#Compute Cosine Similarity between courses
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def recommend_course(student_interest):

    student_vec = tfidf.transform([student_interest])
    # Compute similarity against all courses
    sim_scores = cosine_similarity(student_vec, tfidf_matrix).flatten()
    # Get the index of the best matching course
    best_idx = sim_scores.argmax()

    return {
        "recommended_course": courses_df.loc[best_idx, 'course_name'],
        "category": courses_df.loc[best_idx, 'category'],
        "similarity_score": round(sim_scores[best_idx] * 100, 2)
    }

# Testing
test_query = "Penetration Testing and React"
print(f"Test Query: '{test_query}'")
print("Recommendation Result:", recommend_course(test_query))

# 6. Save the vectorizer and course dataframe for your FastAPI service later
joblib.dump((tfidf, cosine_sim, courses_df), 'course_recommendation_model.pkl')
print("Course Recommendation engine saved as 'course_recommendation_model.pkl'")

Test Query: 'Penetration Testing and React'
Recommendation Result: {'recommended_course': 'Advanced Penetration Testing', 'category': 'Cybersecurity', 'similarity_score': np.float64(91.9)}
Course Recommendation engine saved as 'course_recommendation_model.pkl'


In [14]:

unique_courses_df = df.groupby('course_name').agg({
    'category': 'first',
    'interests': lambda x: ' '.join(set(x))
}).reset_index()

unique_courses_df['combined_features'] = unique_courses_df['category'].fillna('') + ' ' + unique_courses_df['interests'].fillna('')


tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(unique_courses_df['combined_features'])

# Save the lightweight model with unique courses
import joblib
joblib.dump((tfidf, unique_courses_df), 'course_recommendation_model_light.pkl')
print("Saved unique lightweight course recommendation model successfully!")

Saved unique lightweight course recommendation model successfully!


In [15]:
import joblib

# Instead of saving the heavy cosine_sim matrix, save only tfidf vectorizer and courses_df
joblib.dump((tfidf, courses_df), 'course_recommendation_model_light.pkl')
print(" lightweight course recommendation model successfully!")

 lightweight course recommendation model successfully!


Student profiling

In [16]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import joblib

df = pd.read_csv('ed38976-edd_cleaned.csv')

#  Extracting and inspecting student text profiles
df['student_profile_text'] = df['interests'].fillna('') + " | Learning Style: " + df['learning_style'].fillna('')

#Use CountVectorizer to extract key tech terms and skills from student interests
vectorizer = CountVectorizer(stop_words='english', max_features=50)
X_text = vectorizer.fit_transform(df['student_profile_text'])

# Get the top keywords/skills extracted across all students
feature_names = vectorizer.get_feature_names_out()
print(f"Top 20 Extracted Tech/Learning Keywords from Student Profiles:\n{list(feature_names[:20])}")

# Define an NLP Profiling function for incoming student queries or records
def generate_student_profile(student_row_index):
    row = df.iloc[student_row_index]

    # rule-based or keyword-based profiling logic derived from NLP processing
    interests = str(row['interests'])
    learning_style = str(row['learning_style'])
    experience = str(row['experience_level'])

    # Categorize primary track based on keywords in interests
    if any(kw in interests.lower() for kw in ['react', 'html', 'css', 'javascript', 'web']):
        primary_track = "Fullstack Web Development"
    elif any(kw in interests.lower() for kw in ['ai', 'machine learning', 'neural', 'python', 'nlp', 'computer vision']):
        primary_track = "AI & Machine Learning Engineering"
    elif any(kw in interests.lower() for kw in ['cybersecurity', 'hacking', 'penetration']):
        primary_track = "Cybersecurity & Ethical Hacking"
    else:
        primary_track = "Cloud & Big Data Architecture"

    return {
        "student_id": row['student_id'],
        "experience_level": experience,
        "learning_style": learning_style,
        "extracted_interests": interests,
        "assigned_career_profile": primary_track
    }

# Testing the NLP profiler on the first student in the dataset
print("\nSample Generated Student Profile:")
print(generate_student_profile(0))

# Save the vectorizer and profiling logic or artifacts
joblib.dump(vectorizer, 'student_profiling_vectorizer.pkl')
print(" Vectorizer saved as 'student_profiling_vectorizer.pkl'")

Top 20 Extracted Tech/Learning Keywords from Student Profiles:
['ai', 'analysis', 'architecture', 'auditory', 'aws', 'azure', 'big', 'blockchain', 'cloud', 'computer', 'computing', 'css', 'cyber', 'cybersecurity', 'data', 'deep', 'development', 'ethical', 'google', 'hacking']

Sample Generated Student Profile:
{'student_id': '9c19eea1-3955-4b62-90c4-379f6cd2edf7', 'experience_level': 'Beginner', 'learning_style': 'Kinesthetic', 'extracted_interests': 'Penetration Testing, React, Natural Language Processing', 'assigned_career_profile': 'Fullstack Web Development'}
 Vectorizer saved as 'student_profiling_vectorizer.pkl'


Sales Forecasting

The particular dataset didn't contained course fee , so used some demo fees like Beginner:45k
Intermediate:55k
advanced:65k

In [17]:
import pandas as pd
import numpy as np
import joblib
from statsmodels.tsa.arima.model import ARIMA

df = pd.read_csv('ed38976-edd_cleaned.csv')


np.random.seed(42)
date_range = pd.date_range(start='2024-01-01', end='2026-03-01', freq='D')
df['simulated_date'] = np.random.choice(date_range, size=len(df))

# Assigning a mock course fee based on category or difficulty
fee_mapping = {'Beginner': 45000, 'Intermediate': 55000, 'Advanced': 65000}
df['revenue'] = df['difficulty_level'].map(fee_mapping).fillna(50000)

# Aggregate data monthly to create a time-series dataset
df['month'] = pd.to_datetime(df['simulated_date']).dt.to_period('M')
monthly_sales = df.groupby('month')['revenue'].sum().reset_index()
monthly_sales['month'] = monthly_sales['month'].dt.to_timestamp()
monthly_sales = monthly_sales.sort_values('month')

# Trained an ARIMA Time-Series Forecasting Model
ts_data = monthly_sales.set_index('month')['revenue']

# Fit ARIMA model (p, d, q parameters)
model = ARIMA(ts_data, order=(1, 1, 1))
forecast_model_fit = model.fit()

# Forecasting the next 3 months of revenue
forecast_steps = 3
forecast = forecast_model_fit.forecast(steps=forecast_steps)

print(f"Historical Monthly Sales Data Points: {len(ts_data)}")
print("\n--- Revenue Forecast for Next 3 Months ---")
print(forecast)

# Save the trained time-series model artifact
joblib.dump(forecast_model_fit, 'sales_forecasting_model.pkl')
print("\nSales Forecasting Model saved successfully as 'sales_forecasting_model.pkl'")

Historical Monthly Sales Data Points: 27

--- Revenue Forecast for Next 3 Months ---
2026-04-01    3.973010e+06
2026-05-01    2.249221e+06
2026-06-01    3.313108e+06
Freq: MS, Name: predicted_mean, dtype: float64

Sales Forecasting Model saved successfully as 'sales_forecasting_model.pkl'


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Us